In [1]:
import os
import sys
import numpy as np
import subprocess
from pathlib import Path
from shutil import which

In [2]:
# 0..21 から 21(=index 20) を除くチーム一覧
# prep: 1-22 (21 is missing)
# contest: 1-20,22-24
def get_teams():
    arr = np.arange(1, 25)
    return np.delete(arr, 20)  # 21 をスキップ（コンテスト仕様に合わせる）

In [3]:
# testing for team 22 only
def get_teams():
    arr = np.arange(22, 23)
    return arr  

In [4]:
def loop_for_all_teams(
    command_template,
    *,
    dry_run=False,
    strict=True,
    continue_on_error=False,
    cwd=None
):
    """
    command_template: ['python', 'attack/attack_Ci.py', '...{id:02d}...', ...] のようなリスト
    dry_run: True -> 実行せず展開コマンドのみ表示
    strict: True -> {id:02d} が一つも無ければ例外
    continue_on_error: True -> 失敗しても次の team へ。False -> そこで中断
    cwd: サブプロセスの作業ディレクトリ（attack/ ディレクトリの相対パス解決に使える）
    """
    id_indices = [i for i, arg in enumerate(command_template)
                  if isinstance(arg, str) and "{id:02d}" in arg]
    if strict and not id_indices:
        raise ValueError(f"No {{id:02d}} placeholder found in: {command_template}")

    # 'python' を実行ファイルに置換（環境ズレ回避）
    cmd0 = command_template[:]
    if cmd0 and cmd0[0] in ("python", "python3"):
        cmd0[0] = sys.executable

    # 事前: 実行ファイルの存在チェック（python 以外の最初の実体コマンドにも対応）
    exe = cmd0[0]
    if os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    for team in get_teams():
        cmd = cmd0[:]
        for ind in id_indices:
            cmd[ind] = cmd[ind].format(id=team)

        # 簡易プリフライト: 既知の入力系ファイルっぽい引数を存在確認
        # （.csv, .json かつ -o/--out* ではない位置を対象にする）
        def is_out_flag(i):
            return isinstance(cmd[i-1], str) and (
                cmd[i-1] in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
                or cmd[i-1].startswith("--out")
            )

        missing_inputs = []
        for i, a in enumerate(cmd):
            if isinstance(a, str) and (a.endswith(".csv") or a.endswith(".json")):
                if not is_out_flag(i):  # 出力ではなく入力と推定
                    apath = a if cwd is None else os.path.join(cwd, a)
                    if not os.path.exists(apath):
                        missing_inputs.append(a)

        print(">>", " ".join(cmd))
        if missing_inputs:
            msg = f"[team {team}] Missing input files: {missing_inputs}"
            if continue_on_error:
                print("!!", msg)
                continue
            else:
                raise FileNotFoundError(msg)

        if dry_run:
            continue

        try:
            # 標準出力・標準エラーを取得して、失敗時に見せる
            completed = subprocess.run(
                cmd, check=True, cwd=cwd,
                capture_output=True, text=True
            )
            if completed.stdout:
                print(completed.stdout.strip())
        except subprocess.CalledProcessError as e:
            print(f"\n[ERROR] team {team} command failed with code {e.returncode}")
            if e.stdout:
                print("--- stdout ---")
                print(e.stdout.strip())
            if e.stderr:
                print("--- stderr ---")
                print(e.stderr.strip())
            if not continue_on_error:
                raise

In [5]:
def loop_for_all_teams_and_variants(
    command_template,
    variants,
    *,
    dry_run=False,
    strict=True,
    continue_on_error=False,
    cwd=None
):
    """
    command_template: ['python', ..., 'in/BB{id:02d}_{variant}.csv', ...] のようなリスト
    variants: ['1', '2', '3'] など
    """
    id_indices = [i for i, arg in enumerate(command_template)
                  if isinstance(arg, str) and "{id:02d}" in arg]
    variant_indices = [i for i, arg in enumerate(command_template)
                       if isinstance(arg, str) and "{variant}" in arg]
    if strict and not (id_indices or variant_indices):
        raise ValueError("No {id:02d} or {variant} placeholder found in: {}".format(command_template))

    cmd0 = command_template[:]
    if cmd0 and cmd0[0] in ("python", "python3"):
        cmd0[0] = sys.executable

    exe = cmd0[0]
    if os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    for team in get_teams():
        for variant in variants:
            cmd = []
            for arg in cmd0:
                if isinstance(arg, str):
                    cmd.append(arg.format(id=team, variant=variant))
                else:
                    cmd.append(arg)

            def is_out_flag(i):
                return isinstance(cmd[i-1], str) and (
                    cmd[i-1] in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
                    or cmd[i-1].startswith("--out")
                )

            missing_inputs = []
            for i, a in enumerate(cmd):
                if isinstance(a, str) and (a.endswith(".csv") or a.endswith(".json")):
                    if not is_out_flag(i):
                        apath = a if cwd is None else os.path.join(cwd, a)
                        if not os.path.exists(apath):
                            missing_inputs.append(a)

            print(">>", " ".join(cmd))
            if missing_inputs:
                msg = f"[team {team} variant {variant}] Missing input files: {missing_inputs}"
                if continue_on_error:
                    print("!!", msg)
                    continue
                else:
                    raise FileNotFoundError(msg)

            if dry_run:
                continue

            try:
                completed = subprocess.run(
                    cmd, check=True, cwd=cwd,
                    capture_output=True, text=True
                )
                if completed.stdout:
                    print(completed.stdout.strip())
            except subprocess.CalledProcessError as e:
                print(f"\n[ERROR] team {team} variant {variant} command failed with code {e.returncode}")
                if e.stdout:
                    print("--- stdout ---")
                    print(e.stdout.strip())
                if e.stderr:
                    print("--- stderr ---")
                    print(e.stderr.strip())
                if not continue_on_error:
                    raise

In [6]:
# ????????????????????????????????????
project_root_override = None  # ?: r"C:\work\pwscup2025" ????????????????

def find_project_root(start: Path) -> Path:
    markers = ("requirements.txt", "GUIDE_FOR_BEGINNERS.md")
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    return start

if project_root_override:
    project_root = Path(project_root_override).expanduser().resolve()
else:
    project_root = find_project_root(Path.cwd())

os.chdir(project_root)
print(f"Working directory set to: {Path.cwd()}")

Working directory set to: /home/kikuchih/pwscup2025-scripts


In [7]:
## Create input folder and Place "Original Datasets(B**/BB**)".
### create "in/"
### place B22_1, etc. in "in/"
if not os.path.exists("in"):
    os.makedirs("in")

In [8]:
prep_original = "B"
contest_original = "BB"
prep_anon = "C"
contest_anon = "CC"
prep_model = "D"
contest_model = "DD"

In [9]:
# choose mode(prep or contest)
mode = "prep"

if mode == "prep":
    mode_original = prep_original
    mode_anon = prep_anon
    mode_model = prep_model
    variants = ['3']
else:
    mode_original = contest_original
    mode_anon = contest_anon
    mode_model = contest_model
    variants = ['1', '2', '3']

In [10]:
## Creation of Ci with original sample code
Ci_anonymization_original = [
    "python", "anonymization/ano_fixed.py",
    f"in/{mode_original}" + "{id:02d}_{variant}.csv",
    "-o", 
    f"out_anonymized/{mode_anon}" + "{id:02d}_{variant}.csv",
    "--seed", "42"
]
loop_for_all_teams_and_variants(Ci_anonymization_original, variants)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/ano_fixed.py in/B22_3.csv -o out_anonymized/C22_3.csv --seed 42


In [11]:
### evaluation
eval_original = [
    "python", "evaluation/eval_all_fixed.py",
    f"in/{mode_original}" + "{id:02d}_{variant}.csv",
    f"out_anonymized/{mode_anon}" + "{id:02d}_{variant}.csv",
    "-o", f"out_eval/{mode_anon}" + "{id:02d}_{variant}_eval.txt"
]
loop_for_all_teams_and_variants(eval_original, variants)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3.csv -o out_eval/C22_3_eval.txt
stats_diff max_abs: 0.2162671256181917
LR_asthma_diff max_abs: 0.9954029273019501
KW_IND_diff max_abs: 0.1534746161192847
Ci utility: 48.371764106847635 / 80


In [85]:
### eval_txt to eval_csv
eval_original = [
    "python", "anonymization/evaltxt_to_csv.py",
    f"out_eval/{mode_anon}" + "{id:02d}_{variant}_eval.txt",
    "-o", f"out_eval/{mode_anon}"+"{id:02d}_eval.csv"
]
loop_for_all_teams_and_variants(eval_original, variants, strict=False)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_eval.txt -o out_eval/C22_eval.csv


In [86]:
## Creation of Di with original samples(Bi only)
Di_anonymization_original_Bi = ["python", "analysis/xgbt_train_fixed.py", f"in/{mode_original}"+"{id:02d}_{variant}.csv", "-o", f"out_anonymized/{mode_model}"+"{id:02d}_Bi_{variant}.json"]
loop_for_all_teams_and_variants(Di_anonymization_original_Bi, variants)
print("sample Di-anonymization(Bi only) completed")


>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py in/B22_3.csv -o out_anonymized/D22_Bi_3.json
Validation Accuracy (threshold=0.5): 0.888000
Saved model JSON to: out_anonymized/D22_Bi_3.json
#features: 21
sample Di-anonymization(Bi only) completed


In [87]:
## Creation of Di with original samples(Ci only)
Di_anonymization_original_Ci = ["python", "analysis/xgbt_train_fixed.py", f"out_anonymized/{mode_anon}"+"{id:02d}_{variant}.csv", "-o", f"out_anonymized/{mode_model}"+"{id:02d}_Ci_{variant}.json"]
loop_for_all_teams_and_variants(Di_anonymization_original_Ci, variants)
print("sample Di-anonymization(Ci only) completed")

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py out_anonymized/C22_3.csv -o out_anonymized/D22_Ci_3.json
Validation Accuracy (threshold=0.5): 0.740000
Saved model JSON to: out_anonymized/D22_Ci_3.json
#features: 21
sample Di-anonymization(Ci only) completed


In [88]:
## Creation of Di with original samples(Bi and Ci)
Di_anonymization_original = ["python", "anonymization/gen_Di_fixed.py", f"in/{mode_original}"+"{id:02d}_{variant}.csv", f"out_anonymized/{mode_anon}"+"{id:02d}_{variant}.csv", "-o", f"out_anonymized/{mode_model}"+"{id:02d}_BiCi_{variant}.json"]
loop_for_all_teams_and_variants(Di_anonymization_original, variants)
print("sample Ci-anonymization completed")

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/gen_Di_fixed.py in/B22_3.csv out_anonymized/C22_3.csv -o out_anonymized/D22_BiCi_3.json
accuracy: 0.8974
a Di.json example was saved as out_anonymized/D22_BiCi_3.json
sample Ci-anonymization completed


In [ ]:
# git submodule update --init

In [23]:
## Creation of Ci with Mondrian
k_values = [2,3,4,5]
k_values = [6,7,8,9,10]


for k in k_values:
    mondrian_anonymization_original = [
        "python", "third_party/k-anonymity/anonymize-pws.py",
        "--method", "pwscup2025_mondrian",  # ←ここを正しいmethod名に
        "--k", str(k),
        "--input", f"in/{mode_original}" + "{id:02d}_{variant}.csv",
        "-o", f"out_anonymized/{mode_anon}" + f"{{id:02d}}_{{variant}}_mondrian_k{k}.csv"
    ]
    loop_for_all_teams_and_variants(mondrian_anonymization_original, variants)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python third_party/k-anonymity/anonymize-pws.py --method pwscup2025_mondrian --k 6 --input in/B22_3.csv -o out_anonymized/C22_3_mondrian_k6.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python third_party/k-anonymity/anonymize-pws.py --method pwscup2025_mondrian --k 7 --input in/B22_3.csv -o out_anonymized/C22_3_mondrian_k7.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python third_party/k-anonymity/anonymize-pws.py --method pwscup2025_mondrian --k 8 --input in/B22_3.csv -o out_anonymized/C22_3_mondrian_k8.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python third_party/k-anonymity/anonymize-pws.py --method pwscup2025_mondrian --k 9 --input in/B22_3.csv -o out_anonymized/C22_3_mondrian_k9.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python third_party/k-anonymity/anonymize-pws.py --method pwscup2025_mondrian --k 10 --input in/B22_3.csv -o out_anonymized/C22_3_mondrian_k10.csv


In [24]:
### evaluation
# k_values = [2,3,4,5]

for k in k_values:
    eval_original = [
        "python", "evaluation/eval_all_fixed.py",
        f"in/{mode_original}" + "{id:02d}_{variant}.csv",
        f"out_anonymized/{mode_anon}" + f"{{id:02d}}_{{variant}}_mondrian_k{k}.csv",
        "-o", f"out_eval/{mode_anon}" + f"{{id:02d}}_{{variant}}_mondrian_k{k}_eval.txt"
    ]
    loop_for_all_teams_and_variants(eval_original, variants)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3_mondrian_k6.csv -o out_eval/C22_3_mondrian_k6_eval.txt
stats_diff max_abs: 0.2009082873171893
LR_asthma_diff max_abs: 0.9356853717866616
KW_IND_diff max_abs: 0.09578134641077635
Ci utility: 51.33433414336368 / 80
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3_mondrian_k7.csv -o out_eval/C22_3_mondrian_k7_eval.txt
stats_diff max_abs: 0.23685328382897142
LR_asthma_diff max_abs: 0.9356853717866616
KW_IND_diff max_abs: 0.10540305848516329
Ci utility: 49.704100041404644 / 80
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3_mondrian_k8.csv -o out_eval/C22_3_mondrian_k8_eval.txt
stats_diff max_abs: 0.26321586821019216
LR_asthma_diff max_abs: 0.9999999849108406
KW_IND_diff max_abs: 0.10633647244620636
Ci utility: 47.34463612445137 / 80
>

In [25]:
### eval_txt to eval_csv
# k_values = [2,3,4,5]

for k in k_values:
    eval_original = [
        "python", "anonymization/evaltxt_to_csv.py",
        f"out_eval/{mode_anon}" + f"{{id:02d}}_{{variant}}_mondrian_k{k}_eval.txt",
        "-o", f"out_eval/{mode_anon}" + f"{{id:02d}}_mondrian_k{k}_eval.csv"
    ]
    loop_for_all_teams_and_variants(eval_original, variants, strict=False)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_mondrian_k6_eval.txt -o out_eval/C22_mondrian_k6_eval.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_mondrian_k7_eval.txt -o out_eval/C22_mondrian_k7_eval.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_mondrian_k8_eval.txt -o out_eval/C22_mondrian_k8_eval.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_mondrian_k9_eval.txt -o out_eval/C22_mondrian_k9_eval.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_mondrian_k10_eval.txt -o out_eval/C22_mondrian_k10_eval.csv


In [14]:
## DataSynthesizer (複数epsilon/k対応)
# epsilon_list = [1.0, 2.0, ]  # 必要な値に変更
# epsilon_list =  [round(x * 0.1, 1) for x in range(1, 10)] + list(range(1, 21)) # 必要な値に変更
epsilon_list = [0.1, 0.5, 1.0, 3, 5, 10, 15, 20, 25, 30, 40, 50]
k_list = [3]            # 必要な値に変更
ds_mode = "correlated_attribute_mode"  # 必要に応じて変更(['correlated_attribute_mode', 'independent_attribute_mode', 'random_mode'])

for epsilon in epsilon_list:
    for k in k_list:
        DataSynthesizer_anonymization_original = [
            "python",
            "anonymization/ano_DataSynthesizer.py",
            f"in/{mode_original}" + "{id:02d}_{variant}.csv",
            "-o",
            f"out_anonymized/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_{ds_mode}_e{epsilon}_k{k}.csv",
            "--epsilon", str(epsilon),
            "--k", str(k),
            "--mode", f"{ds_mode}",  # 必要に応じて変更(['correlated_attribute_mode', 'independent_attribute_mode', 'random_mode'])
            "--num_tuples", "10000",
            "--seed", "42",
            # "--desc",  f"in/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_e{epsilon}_k{k}.json",
            # "--edges", f"in/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_e{epsilon}_k{k}.pkl",
        ]
        loop_for_all_teams_and_variants(DataSynthesizer_anonymization_original, variants)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/ano_DataSynthesizer.py in/B22_3.csv -o out_anonymized/C22_3_ds_correlated_attribute_mode_e0.1_k3.csv --epsilon 0.1 --k 3 --mode correlated_attribute_mode --num_tuples 10000 --seed 42
================ Constructing Bayesian Network (BN) ================
Adding ROOT ETHNICITY
Adding attribute num_immunizations
Adding attribute mean_weight
Adding attribute obesity_flag
Adding attribute asthma_flag
Adding attribute AGE
Adding attribute encounter_count
Adding attribute GENDER
Adding attribute mean_diastolic_bp
Adding attribute stroke_flag
Adding attribute depression_flag
Adding attribute RACE
Adding attribute mean_bmi
Adding attribute mean_systolic_bp
Adding attribute num_procedures
Adding attribute num_medications
Adding attribute num_allergies
Adding attribute num_devices
========================== BN constructed ==========================
Constructed Bayesian network:
    num_immunizations has parents ['ETHNICITY'].
   

In [18]:
### DataFix (型修正)
# epsilon_list = [1.0, 2.0, ]  # 必要な値に変更
# epsilon_list =  [round(x * 0.1, 1) for x in range(1, 10)] + list(range(1, 21)) # 必要な値に変更
# k_list = [3]            # 必要な値に変更
# ds_mode = "correlated_attribute_mode"  # 必要に応じて変更(['correlated_attribute_mode', 'independent_attribute_mode', 'random_mode'])

for epsilon in epsilon_list:
    for k in k_list:
        DataFix_anonymization_original = [
            "python",
            "anonymization/datafix.py",
            f"in/{mode_original}" + "{id:02d}_{variant}.csv",  # 元データ
            f"out_anonymized/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_{ds_mode}_e{epsilon}_k{k}.csv",  # 型修正対象
            "-o",
            f"out_anonymized/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_{ds_mode}_e{epsilon}_k{k}_fixed.csv",  # 出力先
        ]
        loop_for_all_teams_and_variants(DataFix_anonymization_original, variants)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/datafix.py in/B22_3.csv out_anonymized/C22_3_ds_correlated_attribute_mode_e0.1_k3.csv -o out_anonymized/C22_3_ds_correlated_attribute_mode_e0.1_k3_fixed.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/datafix.py in/B22_3.csv out_anonymized/C22_3_ds_correlated_attribute_mode_e0.5_k3.csv -o out_anonymized/C22_3_ds_correlated_attribute_mode_e0.5_k3_fixed.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/datafix.py in/B22_3.csv out_anonymized/C22_3_ds_correlated_attribute_mode_e1.0_k3.csv -o out_anonymized/C22_3_ds_correlated_attribute_mode_e1.0_k3_fixed.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/datafix.py in/B22_3.csv out_anonymized/C22_3_ds_correlated_attribute_mode_e3_k3.csv -o out_anonymized/C22_3_ds_correlated_attribute_mode_e3_k3_fixed.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/datafix.py in/B22_3.csv ou

In [19]:
### evaluation
# epsilon_list = [1.0, 2.0, ]  # 必要な値に変更
# epsilon_list =  [round(x * 0.1, 1) for x in range(1, 10)] + list(range(1, 21)) # 必要な値に変更
# k_list = [3]            # 必要な値に変更
# ds_mode = "correlated_attribute_mode"  # 必要に応じて変更(['correlated_attribute_mode', 'independent_attribute_mode', 'random_mode'])

for epsilon in epsilon_list:
    for k in k_list:
        eval_original = [
            "python", "evaluation/eval_all_fixed.py",
            f"in/{mode_original}" + "{id:02d}_{variant}.csv",
            f"out_anonymized/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_{ds_mode}_e{epsilon}_k{k}_fixed.csv",
            "-o", f"out_eval/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_{ds_mode}_e{epsilon}_k{k}_fixed_eval.txt"
        ]
        loop_for_all_teams_and_variants(eval_original, variants)



>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3_ds_correlated_attribute_mode_e0.1_k3_fixed.csv -o out_eval/C22_3_ds_correlated_attribute_mode_e0.1_k3_fixed_eval.txt
stats_diff max_abs: 0.9541652343302395
LR_asthma_diff max_abs: 0.933603529596094
KW_IND_diff max_abs: 0.9634653168940477
Ci utility: 3.8920136969875863 / 80
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3_ds_correlated_attribute_mode_e0.5_k3_fixed.csv -o out_eval/C22_3_ds_correlated_attribute_mode_e0.5_k3_fixed_eval.txt
stats_diff max_abs: 0.9541652343302395
LR_asthma_diff max_abs: 0.8966283888406349
KW_IND_diff max_abs: 0.9804621883887007
Ci utility: 4.291579082203709 / 80
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3_ds_correlated_attribute_mode_e1.0_k3_fixed.csv -o out_eval/C22_3_ds_correlated_attribute_mode_e

In [20]:
### eval_txt to eval_csv
# epsilon_list = [1.0, 2.0, ]  # 必要な値に変更
# epsilon_list =  [round(x * 0.1, 1) for x in range(1, 10)] + list(range(1, 21)) # 必要な値に変更
# k_list = [3]            # 必要な値に変更
# ds_mode = "correlated_attribute_mode"  # 必要に応じて変更(['correlated_attribute_mode', 'independent_attribute_mode', 'random_mode'])

for epsilon in epsilon_list:
    for k in k_list:
        eval_original = [
            "python", "anonymization/evaltxt_to_csv.py",
            f"out_eval/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_{ds_mode}_e{epsilon}_k{k}_fixed_eval.txt",
            "-o", f"out_eval/{mode_anon}" + f"{{id:02d}}_ds_{ds_mode}_fixed_eval.csv"
        ]
        loop_for_all_teams_and_variants(eval_original, variants, strict=False)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_ds_correlated_attribute_mode_e0.1_k3_fixed_eval.txt -o out_eval/C22_ds_correlated_attribute_mode_fixed_eval.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_ds_correlated_attribute_mode_e0.5_k3_fixed_eval.txt -o out_eval/C22_ds_correlated_attribute_mode_fixed_eval.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_ds_correlated_attribute_mode_e1.0_k3_fixed_eval.txt -o out_eval/C22_ds_correlated_attribute_mode_fixed_eval.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_ds_correlated_attribute_mode_e3_k3_fixed_eval.txt -o out_eval/C22_ds_correlated_attribute_mode_fixed_eval.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_ds_correlated_attribute_mode_e5_k3_fixe